# Lab 1 · Build the Resident 360 — medallion end-to-end

**One notebook, one Lakehouse, three schemas.** You'll build the full **Bronze → Silver → Gold** medallion for HPB's
Healthy 365 data *inside a single Fabric Lakehouse* (`lh_resident360`), using **schemas** to separate the layers:

| Layer | Schema | What lands here |
|-------|--------|-----------------|
| **Bronze** | `bronze` | Raw app files + the live air-quality API, as-ingested |
| **Silver** | `silver` | Cleaned, conformed facts (CDC, schema evolution, time travel) |
| **Gold**  | `gold`  | The single `resident_360` row per resident, joined with the **mirrored Databricks** estate |

> **Coexistence:** the Databricks estate stays in Databricks — you read it live through the **`hpb_databricks_mirror`**
> you created earlier (zero-copy). Gold joins your Fabric-native Silver with the mirrored Databricks `gold`.

**Before you run:** make sure this notebook has the **`lh_resident360`** Lakehouse attached (Explorer → *Add data
items* → *From OneLake catalog* → the **Lakehouse**, not its SQL endpoint) and that your seven app files are in
**`Files/landing/`**.

Read each section's heading before you run its cells — the markdown explains *what* the step does and *why*.
You can **Run all**, or run section-by-section to watch each layer appear under **Tables**.

## 0 · Setup — create the medallion schemas

Create the three layer schemas in your Lakehouse and set two variables the rest of the notebook uses:
`LANDING` (where your uploaded files live) and `DBX` (your mirrored Databricks gold schema).

In [ ]:
from pyspark.sql import functions as F

# One Lakehouse, three schemas = the medallion layers
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

LANDING = "Files/landing"                 # your seven uploaded Healthy 365 files
DBX     = "hpb_databricks_mirror.gold"    # the mirrored Databricks catalog.schema (zero-copy)

print("Schemas ready: bronze, silver, gold")

### Mirror access smoke test — fail fast

Before building Bronze, verify Spark can read the mirrored Databricks Gold schema. If this fails, stop and tell the facilitator.


In [ ]:
try:
    resident_count = spark.table(f"{DBX}.dim_resident").count()
    assert resident_count == 1500, f"Expected {DBX}.dim_resident to contain 1500 rows, found {resident_count}."
    print(f"Mirror smoke test passed: {DBX}.dim_resident has {resident_count} rows.")
except Exception as exc:
    text = str(exc)
    if "TABLE_OR_VIEW_NOT_FOUND" in text or "cannot be found" in text:
        message = (
            f"Spark cannot find {DBX}.dim_resident.\n\n"
            "This is almost always the MIRROR NAME, and it is fixable by you.\n"
            "The mirrored item's name becomes the Spark catalog name, and the creation\n"
            "wizard defaults to the SOURCE catalog name (hpb_databricks), not the name\n"
            "this notebook expects (hpb_databricks_mirror).\n\n"
            "To fix:\n"
            "  1. In the workspace, check the mirrored item is named exactly\n"
            "     'hpb_databricks_mirror'. Rename it if not.\n"
            "  2. Then STOP AND RESTART this notebook's Spark session. Spark caches the\n"
            "     catalog list per session, so a rename alone will NOT take effect.\n\n"
            f"Original exception: {exc}"
        )
    elif "PERMISSION_DENIED" in text or "AccessDenied" in text or "403" in text:
        message = (
            "Spark found the mirror but was refused access to its files.\n\n"
            "This is a TENANT-LEVEL Unity Catalog / External Data Access setting, not\n"
            "your mistake, and not fixable by retrying or rebuilding the Lakehouse.\n"
            "STOP here and tell the facilitator.\n\n"
            "Observed failure signatures:\n"
            "- PERMISSION_DENIED: External Data Access is disabled for metastore\n"
            "- AccessDeniedException ... 403, HEAD ... _delta_log\n\n"
            f"Original exception: {exc}"
        )
    else:
        message = (
            "Databricks mirror smoke test failed while reading or validating dim_resident.\n\n"
            "Check the mirror item name first, then raise it with the facilitator.\n\n"
            f"Original exception: {exc}"
        )
    print(message)
    raise RuntimeError(message) from exc


## 1 · Bronze — land the Healthy 365 app files

Read all seven uploaded files exactly as they are (no cleaning yet) and write them as `bronze.h365_*` Delta tables.
`residents_reference.csv` is retained as a Bronze reference table for count/key validation; Gold demographics come from the mirrored Databricks `dim_resident`.
Bronze is the *raw* layer: we keep the data faithful to the source, dirty rows and all — we fix those in Silver.

### 0. Residents reference — CSV for key/count validation

This file is uploaded with the app-domain files. It is not joined into Gold because the mirrored Databricks `dim_resident` is the authoritative profile source for this lab.

In [ ]:
res_ref = spark.read.option("header", True).csv(f"{LANDING}/residents_reference.csv")
res_ref.write.mode("overwrite").saveAsTable("bronze.h365_residents_reference")
print("bronze.h365_residents_reference:", spark.table("bronze.h365_residents_reference").count(), "rows")

### 1. Meal logs (diet) — CSV with a few deliberate dirty rows

In [ ]:
meals = spark.read.option("header", True).csv(f"{LANDING}/meal_logs.csv")
meals.write.mode("overwrite").saveAsTable("bronze.h365_meal_logs")
print("bronze.h365_meal_logs:", spark.table("bronze.h365_meal_logs").count(), "rows")

### 2. Event & class bookings

In [ ]:
ev = spark.read.option("header", True).csv(f"{LANDING}/events_bookings.csv")
ev.write.mode("overwrite").saveAsTable("bronze.h365_event_bookings")
print("bronze.h365_event_bookings:", spark.table("bronze.h365_event_bookings").count(), "rows")

### 3. Programme enrolments (Healthier SG, EDSH, I Quit, …)

In [ ]:
pr = spark.read.option("header", True).csv(f"{LANDING}/programme_enrolments.csv")
pr.write.mode("overwrite").saveAsTable("bronze.h365_programme_enrolments")
print("bronze.h365_programme_enrolments:", spark.table("bronze.h365_programme_enrolments").count(), "rows")

### 4. Healthpoints ledger — JSON (multiline array)

In [ ]:
hp = spark.read.option("multiline", True).json(f"{LANDING}/rewards_healthpoints.json")
hp.write.mode("overwrite").saveAsTable("bronze.h365_rewards")
print("bronze.h365_rewards:", spark.table("bronze.h365_rewards").count(), "rows")

### 5. eVoucher redemptions (Healthpoints → merchant vouchers)

In [ ]:
vr = spark.read.option("header", True).csv(f"{LANDING}/evoucher_redemptions.csv")
vr.write.mode("overwrite").saveAsTable("bronze.h365_evoucher_redemptions")
print("bronze.h365_evoucher_redemptions:", spark.table("bronze.h365_evoucher_redemptions").count(), "rows")

### 6. Challenge participation (National Steps Challenge, Eat Drink Shop Healthy, …)

In [ ]:
ch = spark.read.option("header", True).csv(f"{LANDING}/challenges.csv")
ch.write.mode("overwrite").saveAsTable("bronze.h365_challenges")
print("bronze.h365_challenges:", spark.table("bronze.h365_challenges").count(), "rows")

### 7. Verify — seven uploaded-file bronze tables landed

> **Note:** the mirrored Databricks estate (`hpb_databricks_mirror.gold.*`) is read directly — no copy into bronze needed. The next notebook adds an **external API** source.

In [ ]:
for t in ["h365_residents_reference", "h365_meal_logs", "h365_event_bookings", "h365_programme_enrolments",
          "h365_rewards", "h365_evoucher_redemptions", "h365_challenges"]:
    print(f"bronze.{t:32s}", spark.table(f"bronze.{t}").count(), "rows")

### 🧹 Try Data Wrangler on the dirty meal logs

`bronze.h365_meal_logs` has a few **deliberately dirty rows** (blank `calories`, an invalid date). Before we fix them
in code (Silver, next), experience **Data Wrangler** — Fabric's no-code data-cleaning tool that *writes the code for you*:

1. In the notebook toolbar, open the **Data** menu (or the Explorer's `bronze.h365_meal_logs` ⋯) → **Open in Data Wrangler**
   (choose the Spark/pandas option). A grid opens with per-column summary stats.
2. In the **Operations** panel try, for example: **Drop missing values** on `calories`, then **Change column type**
   to cast `calories` to a number. Each click updates a live preview *and* records a step.
3. Click **Add code to notebook** — Data Wrangler drops the generated cleaning code into a new cell. Read it: it's the
   same logic you'll see hand-written in Silver below.

> You don't have to keep the generated cell — the point is to *see* how Data Wrangler turns clicks into reproducible
> code. Silver (Section 3) does the authoritative cleaning for the whole pipeline.

## 2 · Bronze — ingest the live air-quality API

Beyond the app files, we enrich the picture with **environment context**: on hazy days residents skip outdoor activity.
This cell calls the public **data.gov.sg** PSI API and writes `bronze.env_air_quality` (with a safe fallback if the API
is unreachable). PSI is a regional haze-context proxy only — it stands in for ambient outdoor conditions by region,
not a resident's individual route, indoor exposure, or health measurement.

In [ ]:
import requests, datetime
from pyspark.sql import Row, functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

### 1. Call the data.gov.sg PSI API

In [ ]:
url = "https://api.data.gov.sg/v1/environment/psi"
try:
    resp = requests.get(url, timeout=30).json()
    items = resp["items"][0]["readings"]["psi_twenty_four_hourly"]
    ts = resp["items"][0]["timestamp"]
    src = 'live API'
except Exception as e:
    # Offline fallback so the lab always runs
    print("API unreachable, using illustrative fallback PSI values:", str(e)[:120])
    items = {'west': 55, 'east': 48, 'central': 52, 'north': 60, 'south': 50}
    ts = datetime.datetime.now().isoformat()
    src = 'illustrative fallback'
print("PSI source:", src, "| timestamp:", ts)

### 2. Map API regions → our five HPB regions and write bronze

In [ ]:
region_map = {"west":"West","east":"East","central":"Central","north":"North","south":"North-East"}
rows = [Row(reading_ts=ts, region_api=k, psi_24h=int(v), region=region_map.get(k, k.title()))
        for k, v in items.items()]
air = spark.createDataFrame(rows)
air.write.mode("overwrite").saveAsTable("bronze.env_air_quality")
print("bronze.env_air_quality:", spark.table("bronze.env_air_quality").count(), "rows")
display(air)

## 3 · Silver — clean & conform

Now the authoritative cleaning. Each Bronze table becomes a tidy `silver.fact_*`: we cast types, drop the invalid
rows, and conform naming. Along the way you'll see core **DE features** — **Change Data Feed (CDC)**, **schema
evolution**, and **time travel** — the same things you do in Databricks, here in Fabric.

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

### 1. fact_meal_log — drop invalid dates, cast calories
Bad dates (e.g. `31/06/2026`) become null and are filtered; blank calories stay null.

In [ ]:
meal = (spark.table("bronze.h365_meal_logs")
        .withColumn("log_date", F.to_date("log_date"))
        .filter(F.col("log_date").isNotNull())
        .withColumn("calories", F.col("calories").cast("int"))
        .withColumn("healthier_choice_flag", F.upper(F.trim("healthier_choice_flag"))))
meal.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_meal_log")
print("silver.fact_meal_log:", spark.table("silver.fact_meal_log").count(), "rows")

### 2. fact_event_attendance — cast date, add event occurrence grain, enable **Change Data Feed**

`event_id` is the recurring event series, not a unique occurrence. We create `event_occurrence_id` from event id + date + region so Lab 4 relationships do not collapse repeated events across dates or regions. We also persist a distinct occurrence dimension and an attended-only mapping table.

In [ ]:
ev = (spark.table("bronze.h365_event_bookings")
      .withColumn("event_date", F.to_date("event_date"))
      .withColumn(
          "event_occurrence_id",
          F.concat_ws("|", F.col("event_id"), F.date_format("event_date", "yyyy-MM-dd"), F.col("region"))
      ))

ev.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_event_attendance")
spark.sql("ALTER TABLE silver.fact_event_attendance SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

event_occ = ev.select("event_occurrence_id", "event_id", "event_name", "event_type", "event_date", "region").dropDuplicates()
event_occ.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.dim_event_occurrence")

attended_map = (ev.filter(F.upper(F.trim("attended_flag")) == "Y")
                  .select("resident_id", "event_occurrence_id")
                  .dropDuplicates())
attended_map.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.map_resident_event_attendance")

print("silver.fact_event_attendance:", spark.table("silver.fact_event_attendance").count(), "rows")
print("silver.dim_event_occurrence:", spark.table("silver.dim_event_occurrence").count(), "rows")
print("silver.map_resident_event_attendance:", spark.table("silver.map_resident_event_attendance").count(), "rows")

### 3. fact_programme_enrolment

In [ ]:
pr = (spark.table("bronze.h365_programme_enrolments")
      .withColumn("enrol_date", F.to_date("enrol_date")))
pr.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_programme_enrolment")
print("silver.fact_programme_enrolment:", spark.table("silver.fact_programme_enrolment").count(), "rows")

### 4. fact_rewards — Healthpoints ledger

In [ ]:
rw = (spark.table("bronze.h365_rewards")
      .withColumn("txn_date", F.to_date("txn_date"))
      .withColumn("points_earned", F.col("points_earned").cast("int"))
      .withColumn("points_redeemed", F.col("points_redeemed").cast("int")))
rw.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_rewards")
print("silver.fact_rewards:", spark.table("silver.fact_rewards").count(), "rows")

### 5. fact_evoucher_redemption

In [ ]:
vr = (spark.table("bronze.h365_evoucher_redemptions")
      .withColumn("redeemed_date", F.to_date("redeemed_date"))
      .withColumn("voucher_value_sgd", F.col("voucher_value_sgd").cast("double"))
      .withColumn("healthpoints_spent", F.col("healthpoints_spent").cast("int")))
vr.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_evoucher_redemption")
print("silver.fact_evoucher_redemption:", spark.table("silver.fact_evoucher_redemption").count(), "rows")

### 6. fact_challenge — challenge participation & progress

In [ ]:
ch = (spark.table("bronze.h365_challenges")
      .withColumn("enrol_date", F.to_date("enrol_date"))
      .withColumn("progress_pct", F.col("progress_pct").cast("double")))
ch.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("silver.fact_challenge")
print("silver.fact_challenge:", spark.table("silver.fact_challenge").count(), "rows")

### 7. DE feature demo — schema evolution + time travel
Append one row that adds a **new column** (`logged_via`) with `mergeSchema`, then read history.

In [ ]:
extra = spark.sql("""SELECT
    string('RESIDENT_00001') AS resident_id, date('2026-06-30') AS log_date,
    'Snack' AS meal_type, 'Apple' AS food_item, 52 AS calories,
    'Y' AS healthier_choice_flag, 'App' AS logged_via""")
extra.write.mode("append").option("mergeSchema", True).saveAsTable("silver.fact_meal_log")
display(spark.sql("DESCRIBE HISTORY silver.fact_meal_log").select("version", "timestamp", "operation"))

### 8. Verify — six silver facts plus event occurrence helpers

In [ ]:
for t in ["fact_meal_log", "fact_event_attendance", "dim_event_occurrence", "map_resident_event_attendance",
          "fact_programme_enrolment", "fact_rewards", "fact_evoucher_redemption", "fact_challenge"]:
    print(f"silver.{t:26s}", spark.table(f"silver.{t}").count(), "rows")

## 4 · Gold — the `resident_360` table

The payoff: **one row per resident** that unifies **mirrored Databricks** demographics/activity/screening with your
**Fabric-native Silver** domains (meals, events, programmes, rewards, challenges) and the **air-quality** context —
then derives an `is_disengaged` flag used later in the ML lab.

> This is the coexistence moment: `gold.resident_360` joins `hpb_databricks_mirror.gold.*` (read live from Databricks)
> with your own `silver.*` tables — no copy of the Databricks estate.

In [ ]:
from pyspark.sql import functions as F
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
DBX = "hpb_databricks_mirror.gold"     # your mirrored Databricks catalog.schema

### 1. Demographics + activity + screening — from the mirrored Databricks estate

In [ ]:
dim = spark.table(f"{DBX}.dim_resident")

act = (spark.table(f"{DBX}.daily_activity").groupBy("resident_id").agg(
        F.avg("steps").alias("avg_daily_steps"),
        F.avg("mvpa_minutes").alias("avg_mvpa_min"),
        F.avg("sleep_minutes").alias("avg_sleep_min"),
        F.sum("goal_met").alias("days_goal_met"),
        F.count("*").alias("active_days")))

# latest screening per resident (most recent screening_date)
from pyspark.sql import Window
w = Window.partitionBy("resident_id").orderBy(F.col("screening_date").desc())
scr = (spark.table(f"{DBX}.health_screening")
       .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
       .select("resident_id", F.col("bmi").alias("latest_bmi"),
               F.col("systolic_bp").alias("latest_systolic"),
               F.col("risk_band").alias("screening_risk")))

### 2. Fabric-native domains — from silver

In [ ]:
meal = (spark.table("silver.fact_meal_log").groupBy("resident_id").agg(
            F.count("*").alias("meal_logs"),
            F.avg("calories").alias("avg_calories"),
            F.avg(F.when(F.col("healthier_choice_flag") == "Y", 1).otherwise(0)).alias("pct_healthier_choice")))

evt = (spark.table("silver.fact_event_attendance").groupBy("resident_id").agg(
            F.sum(F.when(F.col("attended_flag") == "Y", 1).otherwise(0)).alias("events_attended"),
            F.count("*").alias("events_booked")))

prog = (spark.table("silver.fact_programme_enrolment").groupBy("resident_id").agg(
            F.count("*").alias("programmes_enrolled"),
            F.sum(F.when(F.col("status") == "Dropped", 1).otherwise(0)).alias("programmes_dropped")))

rew = (spark.table("silver.fact_rewards").groupBy("resident_id").agg(
            F.sum("points_earned").alias("healthpoints_earned"),
            F.sum("points_redeemed").alias("healthpoints_redeemed")))

vou = (spark.table("silver.fact_evoucher_redemption").groupBy("resident_id").agg(
            F.count("*").alias("vouchers_redeemed"),
            F.sum("voucher_value_sgd").alias("voucher_value_sgd")))

cha = (spark.table("silver.fact_challenge").groupBy("resident_id").agg(
            F.sum(F.when(F.col("status") == "Active", 1).otherwise(0)).alias("challenges_active"),
            F.avg("progress_pct").alias("avg_challenge_progress")))

### 2b. Regional environmental context — air quality (PSI)
Attach a **regional PSI context proxy** so the 360 can discuss haze as a possible outdoor-activity factor. `region_is_hazy = 1` when the region's 24-hour PSI is >= 55. This is not individual exposure, indoor exposure, or a health measurement. The cell reads `bronze.env_air_quality` from this notebook and falls back to illustrative values only if that table is unavailable.

In [ ]:
try:
    airq = (spark.table("bronze.env_air_quality")
                 .groupBy("region").agg(F.round(F.avg("psi_24h")).cast("int").alias("region_psi")))
    assert airq.count() > 0
    print("air quality: from bronze.env_air_quality")
except Exception as e:
    print("bronze.env_air_quality unavailable — using illustrative fallback PSI:", str(e)[:80])
    airq = spark.createDataFrame(
        [("West", 55), ("East", 48), ("Central", 52), ("North", 60), ("North-East", 50)],
        ["region", "region_psi"])
airq.show()

### 3. Join into one Resident 360 row + derive `is_disengaged`
`is_disengaged = 1` when avg steps < 4000 **and** no events attended **and** a dropped programme.
We also attach `region_psi` and a `region_is_hazy` flag (24-hour PSI ≥ 55) for the environmental view.

In [ ]:
r360 = (dim.join(act, "resident_id", "left").join(scr, "resident_id", "left")
           .join(meal, "resident_id", "left").join(evt, "resident_id", "left")
           .join(prog, "resident_id", "left").join(rew, "resident_id", "left")
           .join(vou, "resident_id", "left").join(cha, "resident_id", "left")
           .join(airq, "region", "left")
           .fillna({"events_attended": 0, "programmes_dropped": 0, "avg_daily_steps": 0,
                    "vouchers_redeemed": 0, "healthpoints_earned": 0})
           .withColumn("screening_risk", F.coalesce("screening_risk", F.lit("Not Screened")))
           .withColumn("region_psi", F.coalesce("region_psi", F.lit(50)))
           .withColumn("region_is_hazy", F.when(F.col("region_psi") >= 55, 1).otherwise(0))
           .withColumn("is_disengaged",
               F.when((F.col("avg_daily_steps") < 4000) & (F.col("events_attended") < 1)
                      & (F.col("programmes_dropped") > 0), 1).otherwise(0)))

r360.write.mode("overwrite").option("mergeSchema", True).saveAsTable("gold.resident_360")
print("gold.resident_360:", spark.table("gold.resident_360").count(), "rows")

### 4. Verify — profile + disengagement split

In [ ]:
g = spark.table("gold.resident_360")
print("columns:", len(g.columns))
g.groupBy("is_disengaged").count().show()
display(g.select("resident_id","region","region_is_hazy","age_band","avg_daily_steps","events_attended",
                 "programmes_dropped","screening_risk","healthpoints_earned","is_disengaged").limit(20))

### 5. Entity dimensions for the ontology (Lab 4 depends on these)

Lab 4 binds ontology **entity types** to these tables. An ontology entity key must
**uniquely identify each record** it ingests, so an entity cannot be bound to a transactional
table: `gold.resident_360` holds 5 regions across 1,500 rows, and `silver.fact_programme_enrolment`
holds 6 programmes across 3,004. Binding an entity there produces an ontology that looks correct,
instantiates nothing, and answers every question with *"the graph query is failing on the backend"*.

`Resident` and `EventOccurrence` are already at entity grain — `gold.resident_360` is one row per
resident, `silver.dim_event_occurrence` one row per occurrence. These three complete the set.


In [ ]:
from pyspark.sql import functions as F

# One row per region, per programme, per challenge - entity grain for Lab 4's ontology.
(spark.table("gold.resident_360").groupBy("region")
    .agg(F.round(F.avg("region_psi")).cast("int").alias("region_psi"),
         F.max("region_is_hazy").cast("int").alias("region_is_hazy"),
         F.count("*").cast("long").alias("resident_count"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_region"))

(spark.table("silver.fact_programme_enrolment").groupBy("programme_name")
    .agg(F.countDistinct("resident_id").cast("long").alias("enrolled_residents"),
         F.sum(F.when(F.col("status") == "Dropped", 1).otherwise(0)).cast("long").alias("dropped_count"),
         F.min("enrol_date").alias("first_enrolment"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_programme"))

(spark.table("silver.fact_challenge").groupBy("challenge_name")
    .agg(F.first("target_metric").alias("target_metric"),
         F.countDistinct("resident_id").cast("long").alias("participant_count"),
         F.round(F.avg("progress_pct"), 2).alias("avg_progress_pct"))
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.dim_challenge"))

# Entity grain is the whole point, so prove it rather than assume it.
for t, key in [("silver.dim_region", "region"),
               ("silver.dim_programme", "programme_name"),
               ("silver.dim_challenge", "challenge_name")]:
    df = spark.table(t)
    n, d = df.count(), df.select(key).distinct().count()
    assert n == d, f"{t}: {n} rows but only {d} distinct {key} - not entity grain"
    print(f"{t:22s} rows={n:<5d} distinct[{key}]={d:<5d} OK")


## 5 · Data-quality gate

A lightweight check that Gold is sane before anything downstream (reports, ML, the data agent) consumes it — the same
idea the metadata-driven framework formalises in Lab 2.

In [ ]:
from pyspark.sql import functions as F

df = spark.table("bronze.h365_meal_logs")
total = df.count()
bad_dates = df.filter(F.to_date("log_date").isNull()).count()
null_cal = df.filter(F.col("calories").isNull() | (F.trim("calories") == "")).count()
bad_pct = round(100.0 * (bad_dates + null_cal) / max(total, 1), 2)
status = "PASS" if bad_pct < 5.0 else "FAIL"
result = {"status": status, "total": total, "bad_dates": bad_dates,
          "null_calories": null_cal, "bad_pct": bad_pct}
print(result)

# A data-quality gate for downstream use. It reports the measured checks and stops the notebook if they fail.
if status == "FAIL":
    raise ValueError("DQ gate FAILED for bronze.h365_meal_logs; fix or investigate before using downstream labs.")
else:
    print("✅ DQ gate PASSED for bronze.h365_meal_logs checks. Lab 2 can record this run's audit status.")


## 6 · Observe how your jobs run — Spark UI, prioritisation & monitoring

Your medallion is built. Now *watch how Spark ran it*. The cells below run a deliberately **skewed** job and then a
**tuned** one so you can compare them in the **Spark UI** and the **Monitoring** hub.

1. Run the **skewed** cell. While it runs, open the cell's **Spark jobs** view (or the **Monitor** hub → this notebook's
   run) and note the single long-running task — everything forced through one partition.
2. Run the **tuned** cell (Adaptive Query Execution + balanced partitions) and compare: more, shorter, parallel tasks.
3. The **resource-prioritisation** note shows how a nightly ETL and ad-hoc queries share the capacity (custom pool /
   Autoscale Billing for Spark).

> **Where to look:** the **Spark UI** is reachable from any running cell's *Spark jobs* link; the **Monitoring** hub
> (left nav → *Monitor*) lists every notebook/pipeline run with duration, status and Spark detail.

In [ ]:
from pyspark.sql import functions as F
DBX = "hpb_databricks_mirror.gold"
act = spark.table(f"{DBX}.daily_activity")
print("activity rows:", act.count())

### 1. Skewed job — everything forced through ONE partition
Run this, then open **Spark UI** (notebook status bar → **Spark UI**) → **Stages**. Note the single long task, shuffle read/write and any spill.

In [ ]:
heavy = (act.repartition(1)                                   # SKEW: single partition
         .groupBy("resident_id")
         .agg(F.avg("steps").alias("avg_steps"),
              F.expr("percentile_approx(steps, 0.9)").alias("p90_steps"),
              F.stddev("steps").alias("sd_steps")))
print("rows:", heavy.count())

### 2. Tuned job — Adaptive Query Execution + balanced partitions
Same result, healthier execution: compare task count and duration in the Spark UI.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")          # AQE
tuned = (act.repartition(8, "resident_id")                    # 8 balanced partitions
         .groupBy("resident_id")
         .agg(F.avg("steps").alias("avg_steps"),
              F.expr("percentile_approx(steps, 0.9)").alias("p90_steps"),
              F.stddev("steps").alias("sd_steps")))
print("rows:", tuned.count())

### 3. Resource prioritisation — nightly ETL vs ad-hoc

On Fabric, **capacity (CU)** replaces cluster sizing. When a scheduled ETL and ad-hoc analysis run at once they share the same capacity, so a heavy ad-hoc job can starve the nightly load.

**How Fabric lets you prioritise / isolate:**
- **Custom (On-Demand) Spark pool + Environment** — give the nightly ETL its own pool so it isn't queued behind ad-hoc sessions.
- **Autoscale Billing for Spark** — move bursty ad-hoc Spark to dedicated serverless billed separately, so it never competes with the capacity that runs production ETL.
- **High-concurrency session sharing** — many light notebooks share one session to save CU for the jobs that matter.

**See it live:** open the facilitator's **Spark Monitoring** KQL dashboard (from the fabric-toolbox accelerator) to compare this heavy application's memory / CPU / shuffle / spill against a tuned run, and read its SparkLens recommendation on whether more resources would actually help.

> **Try it (optional):** re-run cell 1 while a neighbour runs a job on the same capacity, then watch both runs contend in **Monitor → Activities** (left nav). Note that Activities lists **one row per item run** — your notebook appears once as `resident360_medallion_<guid>`, not once per Spark job — so click into a run to see its per-cell Spark detail.

## 7 · Explore with Copilot (agent mode)

Finally, experience **Copilot in the notebook**. Click the **Copilot** button in the toolbar to open the chat, then
switch it to **agent mode** (it can plan and run multiple steps for you, not just suggest one cell).

Try prompts like:
- *"Profile `gold.resident_360`: row count, % disengaged, and average steps by region. Add the results as a new cell."*
- *"Chart average `mvpa_minutes` by `age_band` from `gold.resident_360`."*
- *"Explain what the Silver `fact_meal_log` transformation does and why the date filter matters."*

Watch Copilot **plan → generate → run** and inspect the cells it adds. This is the same assistant you'll lean on when
you build the Copilot-generated **report** next (in the lab README, Task 4).

---

✅ **Done.** Your Lakehouse now has `bronze.*`, `silver.*`, and `gold.resident_360`. Head back to the lab README to
build the **semantic model** and the **Copilot-generated report**.